# Decaimiento de la tasa de aprendizaje

(Ejercicio, 3 puntos posibles)

En los notebooks 21 y 22 vimos que la **tasa de aprendizaje** $\eta$ decide el tamaño de cada paso del descenso por gradiente:

- si $\eta$ es **pequeña**, el algoritmo avanza muy lento;
- si $\eta$ es **grande**, avanza rápido al principio pero, cerca del mínimo, **rebota** de un lado al otro del valle o incluso diverge.

La idea del **decaimiento de la tasa de aprendizaje** (*learning rate decay*) es usar lo mejor de los dos mundos: empezar con una $\eta$ grande para avanzar rápido y **reducirla poco a poco** para afinar cerca del mínimo.

Tres reglas comunes, donde $t$ es el número de iteración y $\eta_0$ la tasa inicial:

| Regla | Fórmula | Idea |
|---|---|---|
| Escalonada (*step*) | $\eta_t = \eta_0 \cdot \gamma^{\lfloor t / s \rfloor}$ | Cada $s$ iteraciones multiplica la tasa por $\gamma$ (por ejemplo, la reduce a la mitad) |
| Inversa en el tiempo | $\eta_t = \dfrac{\eta_0}{1 + k\,t}$ | Baja rápido al inicio y luego cada vez más despacio |
| Exponencial | $\eta_t = \eta_0 \, e^{-k t}$ | Baja de forma suave y continua |

$\lfloor \cdot \rfloor$ significa "redondear hacia abajo" (división entera: `t // s` en Python).

*Notebook complementario del curso (repositorio datahuba/curso-redes-neuronales), en el estilo de los notebooks 21 y 22 del Dr. J. Irving Vasquez.*

In [ ]:
import math
import matplotlib.pyplot as plt
import numpy as np
import nni

## El problema: la función de Beale

Usaremos la misma función del notebook 22. Su mínimo global está en $(3, 0.5)$ y vale $0$. Tiene un valle largo y estrecho: justo donde una tasa de aprendizaje fija suele rebotar.

In [ ]:
problem = nni.ReducedBealeProblem()

x = np.linspace(problem.min_x, problem.max_x, 100)
y = np.linspace(problem.min_y, problem.max_y, 100)
X, Y = np.meshgrid(x, y)
Z = problem.function(X, Y)
Z_log = np.log(Z + 1)   # escala logarítmica para ver mejor el valle

plt.contourf(X, Y, Z_log, levels=100, cmap='viridis')
plt.plot(problem.optimal_params[0], problem.optimal_params[1], 'ro')
plt.title('Función de Beale (escala logarítmica)')
plt.xlabel('x')
plt.ylabel('y')
plt.show()

## Reglas de decaimiento

Implementa las tres reglas de la tabla de arriba. Cada función recibe la tasa inicial `lr0` y la iteración `t`, y devuelve la tasa que corresponde a esa iteración.

In [ ]:
# TODO (1 punto): implementa las tres reglas de decaimiento

def tasa_escalonada(lr0, t, gamma=0.5, cada=100):
    return None

def tasa_inversa(lr0, t, k=0.01):
    return None

def tasa_exponencial(lr0, t, k=0.01):
    return None

In [ ]:
# Comprobación: compara con los valores esperados de la guía
for t in [0, 100, 250]:
    print(f"t = {t:>3}:  escalonada = {tasa_escalonada(0.05, t):.5f}   "
          f"inversa = {tasa_inversa(0.05, t):.5f}   exponencial = {tasa_exponencial(0.05, t):.5f}")

## Optimizador con decaimiento

Partimos del optimizador de descenso por gradiente del paquete `nni` (el mismo del notebook 22). Su método `optimize` llama a `next_params` en cada iteración; nosotros lo cambiamos para que, **antes de dar el paso**, calcule la tasa de la iteración actual.

Recuerda la regla del descenso: $W_{nuevo} = W - \eta_t \, \nabla f(W)$.

In [ ]:
# TODO (1 punto): completa next_params
class DecayOptimizer(nni.GradientDescentOptimizer):
    def __init__(self, lr0=0.05, regla=tasa_escalonada, max_iterations=500, tolerance=1e-6):
        super().__init__(learning_rate=lr0, max_iterations=max_iterations, tolerance=tolerance)
        self.lr0 = lr0
        self.regla = regla
        self.t = 0            # contador de iteraciones
        self.tasas = []       # historial de tasas, para graficarlas
        self.name = f'Decaimiento ({regla.__name__})'

    def next_params(self, gradient, params):
        # 1. calcula la tasa de esta iteración con self.regla(self.lr0, self.t)
        # 2. guárdala en self.tasas y en self.learning_rate
        # 3. aumenta el contador self.t
        # 4. devuelve los nuevos parámetros: param - tasa * gradiente (para cada parámetro)
        return None

In [ ]:
# -- hiperparámetros --
punto_inicial = [1.0, 1.0]
lr0 = 0.05
max_iterations = 500

gradiente = lambda x, y: [problem.gradient_x(x, y), problem.gradient_y(x, y)]

fijo = nni.GradientDescentOptimizer(learning_rate=lr0, max_iterations=max_iterations, tolerance=1e-6)
p_fijo = fijo.optimize(problem.function, gradiente, list(punto_inicial))

decae = DecayOptimizer(lr0=lr0, regla=tasa_escalonada, max_iterations=max_iterations)
p_decae = decae.optimize(problem.function, gradiente, list(punto_inicial))

print(f"Tasa fija {lr0}:       termina en ({p_fijo[0]:.3f}, {p_fijo[1]:.3f}),  f = {problem.function(*p_fijo):.2e}")
print(f"Con decaimiento:     termina en ({p_decae[0]:.3f}, {p_decae[1]:.3f}),  f = {problem.function(*p_decae):.2e}")

In [ ]:
# Trayectorias y tasa de aprendizaje
fig, axs = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, opt in zip(axs[:2], [fijo, decae]):
    h = opt.get_history()
    ax.contourf(X, Y, Z_log, levels=100, cmap='viridis')
    ax.plot([l['params'][0] for l in h], [l['params'][1] for l in h], 'w.-', markersize=4)
    ax.plot(problem.optimal_params[0], problem.optimal_params[1], 'ro')
    ax.set_title(opt.name)
axs[2].plot(decae.tasas)
axs[2].set_title('Tasa de aprendizaje por iteración')
axs[2].set_xlabel('iteración')
plt.tight_layout()
plt.show()

In [ ]:
# TODO (1 punto): experimenta
# 1. Cambia regla=tasa_escalonada por tasa_inversa y por tasa_exponencial. ¿Cuál llega más cerca del mínimo?
# 2. Prueba la tasa fija lr0 = 0.01. ¿Llega igual de cerca en 500 iteraciones?
# 3. Prueba gamma = 0.8 en la regla escalonada (decae más lento).

## Preguntas

- ¿Por qué con una tasa fija de 0.05 el optimizador no termina de llegar al mínimo?
- ¿Qué ventaja tiene empezar con una tasa grande y luego reducirla?
- ¿Qué pasaría si la tasa decae demasiado rápido?